In [1]:
import os
import cv2
import torch
import glob
import gc 

import time
import torch.amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchmetrics.detection.mean_ap import MeanAveragePrecision

from ultralytics import YOLO, RTDETR
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F

import albumentations as A
from albumentations.pytorch import ToTensorV2

import csv
import types

In [2]:
FOLDS_DIR = 'dataset_folds'
NUM_FOLDS = 1
EPOCHS_YOLO = 1
EPOCHS_DETR = 1
EPOCHS_FRCNN = 20
BATCH_SIZE = 8
NUM_CLASSES = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
def free_memory():
    """Агрессивная очистка RAM и VRAM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def on_epoch_end_callback(*args, **kwargs):
    """Коллбек для Ultralytics, вызывается в конце каждой эпохи."""
    free_memory()

In [4]:
class EarlyStoppingMAP:
    """Early Stopping по метрике mAP (максимизация)"""
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_map = 0.0
        self.early_stop = False

    def __call__(self, current_map):
        if current_map > self.best_map + self.min_delta:
            self.best_map = current_map
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

In [5]:
class YoloToFasterRCNNDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, augment=False):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.augment = augment
        
        self.imgs = [img for img in os.listdir(img_dir) 
                    if img.lower().endswith(('.jpg', '.png', '.jpeg'))]
        
        print(f"Dataset: {len(self.imgs)} изображений | Аугментации: {'ON' if augment else 'OFF'}")

        if self.augment:
            self.transforms = A.Compose([
                A.Resize(640, 640),
                A.HorizontalFlip(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.15, rotate_limit=15, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
                A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=15, p=0.3),
                A.Blur(blur_limit=3, p=0.1),
                A.CLAHE(p=0.2),
                # Нормализация [0, 1] необходима для Faster RCNN
                A.ToFloat(max_value=255.0), 
                ToTensorV2()
            ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.2))
        else:
            self.transforms = A.Compose([
                A.ToFloat(max_value=255.0),
                ToTensorV2()
            ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Не удалось прочитать: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape

        lbl_name = os.path.splitext(img_name)[0] + '.txt'
        lbl_path = os.path.join(self.lbl_dir, lbl_name)

        boxes = []
        labels = []

        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) != 5: continue
                    class_id, cx, cy, bw, bh = map(float, parts)

                    # Конвертация YOLO (normalized cx,cy,w,h) -> Pascal VOC (absolute xmin, ymin, xmax, ymax)
                    x1 = (cx - bw / 2) * w
                    y1 = (cy - bh / 2) * h
                    x2 = (cx + bw / 2) * w
                    y2 = (cy + bh / 2) * h

                    # Ограничение координат пределами изображения
                    x1 = max(0.0, min(float(w), x1))
                    y1 = max(0.0, min(float(h), y1))
                    x2 = max(0.0, min(float(w), x2))
                    y2 = max(0.0, min(float(h), y2))

                    if (x2 - x1) > 1.0 and (y2 - y1) > 1.0:
                        boxes.append([x1, y1, x2, y2])
                        labels.append(int(class_id) + 1) # Фон это 0, классы с 1

        # Применение Albumentations
        transformed = self.transforms(image=img, bboxes=boxes, labels=labels)
        img_tensor = transformed['image']
        trans_boxes = transformed['bboxes']
        trans_labels = transformed['labels']

        if len(trans_boxes) > 0:
            boxes = torch.as_tensor(trans_boxes, dtype=torch.float32)
            labels = torch.as_tensor(trans_labels, dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        return img_tensor, target

    def __len__(self):
        return len(self.imgs)

In [6]:
def collate_fn(batch):
    return tuple(zip(*batch))

def get_faster_rcnn_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes + 1)
    return model

In [7]:
def train_faster_rcnn(fold_idx, fold_dir):
    print(f"\n--- [1/4] Подготовка данных Faster R-CNN для Фолда {fold_idx} ---")
    
    train_dataset = YoloToFasterRCNNDataset(
        os.path.join(fold_dir, 'images', 'train'), 
        os.path.join(fold_dir, 'labels', 'train'),
        augment=True
    )
    val_dataset = YoloToFasterRCNNDataset(
        os.path.join(fold_dir, 'images', 'val'), 
        os.path.join(fold_dir, 'labels', 'val'),
        augment=False
    )
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=0, pin_memory=True)
    
    print(f"--- [2/4] Загрузка модели ---")
    model = get_faster_rcnn_model(NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=0.005, 
        momentum=0.9, 
        weight_decay=0.0005
    )
    
    # ВАЖНО: mode='max', так как мы максимизируем mAP
    scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)
    early_stopping = EarlyStoppingMAP(patience=5)
    scaler = torch.amp.GradScaler() if torch.cuda.is_available() else None
    
    print(f"--- [3/4] Обучение с AMP + EarlyStopping (mAP) + Gradient Clipping ---")

    # model.train()
    best_val_map = 0.0
    
    os.makedirs('runs/faster_rcnn', exist_ok=True)   
    
    log_file = f'runs/faster_rcnn/fold_{fold_idx}_metrics.csv'
    with open(log_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_map'])

    for epoch in range(EPOCHS_FRCNN):
        # --- Train ---
        model.train()
        epoch_loss = 0
        for step, (images, targets) in enumerate(train_loader):
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
            
            if scaler:
                scaler.scale(losses).backward()
                scaler.unscale_(optimizer)
            else:
                losses.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            
            epoch_loss += losses.item()
            
            if (step + 1) % 50 == 0 or (step + 1) == len(train_loader):
                print(f"  Эпоха {epoch+1}/{EPOCHS_FRCNN} | Батч {step+1}/{len(train_loader)} | Loss: {losses.item():.4f}")
        
        train_avg_loss = epoch_loss / len(train_loader)
        
        # --- Validation (mAP) ---
        model.eval() # Обязательно для генерации предсказаний (boxes, scores, labels)
        map_metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox').to(DEVICE)
        
        with torch.no_grad():
            for images, targets in val_loader:
                if len(images) == 0: continue
                images = [img.to(DEVICE) for img in images]
                targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
                
                # Получаем предсказания
                outputs = model(images)
                
                # Обновляем метрику mAP
                map_metric.update(outputs, targets)
        
        # Считаем итоговые метрики
        metrics = map_metric.compute()
        val_map = metrics['map'].item() # Можно использовать 'map' для mAP@0.5:0.95
        
        with open(log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch + 1, f"{train_avg_loss:.4f}", f"{val_map:.4f}"])

        print(f"→ Эпоха {epoch+1} | Train Loss: {train_avg_loss:.4f} | Val mAP@95: {val_map:.4f}")
        
        # Scheduler и Early Stopping реагируют на mAP
        scheduler.step(val_map)
        
        # Сохраняем лучшую модель по mAP
        if val_map > best_val_map:
            best_val_map = val_map
            os.makedirs('runs/faster_rcnn', exist_ok=True)
            torch.save(model.state_dict(), f"runs/faster_rcnn/fold_{fold_idx}_best.pth")
            print(f"  [+] Новая лучшая модель сохранена! mAP@95: {best_val_map:.4f}")
            
        if early_stopping(val_map):
            print("Early Stopping активирован (mAP перестал расти)!")
            break
            
        # Очистка памяти метрики
        map_metric.reset()
        del map_metric
    
    # Финальное сохранение (last)
    os.makedirs('runs/faster_rcnn', exist_ok=True)
    torch.save(model.state_dict(), f"runs/faster_rcnn/fold_{fold_idx}_last.pth")
    
    print(f" Фолд {fold_idx} завершён | Лучший Val mAP@95: {best_val_map:.4f}\n")
    
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    free_memory()

In [8]:
def train_yolo11_model(fold_idx, fold_dir):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))
    
    print(f"--- Обучение YOLO11m на Фолде {fold_idx} ---")
    yolo_model = YOLO('yolo11m.pt')
    
    # Коллбек очистки памяти после каждой эпохи
    yolo_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)
    
    yolo_model.train(
        data=yaml_path,
        epochs=EPOCHS_YOLO,
        batch=BATCH_SIZE,
        imgsz=640,
        patience=15,                    # Early Stopping
        project='runs/ensemble_training',
        name=f'yolo11m_fold_{fold_idx}',
        device=0,
        workers=4,
        augment=True,
        
        box=7.5,                        
        cls=1.0,
        dfl=1.5,
        cos_lr=True,    

        fliplr=0.5,         # Отзеркаливание по горизонтали 
        flipud=0.0,         # Отзеркаливание по вертикали 
        degrees=30.0,        # Вращение 
        translate=0.2,      # Сдвиг 
        scale=0.6,          # Масштаб (+-60%)
        shear=20.0,          # Сдвиг/Перекос 
        perspective=0.0001,
        
        hsv_h=0.015,        # Сдвиг оттенка. 
        hsv_s=0.3,          # Насыщенность (70%). 
        hsv_v=0.3,          # Яркость (50%). 
        bgr=0.0,            # Переворот каналов BGR->RGB 

        close_mosaic=10,
        mosaic=0.5,
        mixup=0.15,
        copy_paste=0.3,
        erasing=0.4,
       
        multi_scale=0.25
    )
    
    # Глобальная очистка
    del yolo_model
    free_memory()

In [14]:
def train_yolo26_model(fold_idx, fold_dir):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))
    
    print(f"--- Обучение YOLO11m на Фолде {fold_idx} ---")
    yolo_model = YOLO('yolo26m.pt')
    
    # Коллбек очистки памяти после каждой эпохи
    yolo_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)
    
    yolo_model.train(
        data=yaml_path,
        epochs=EPOCHS_YOLO,
        batch=BATCH_SIZE,
        imgsz=640,
        patience=15,                    # Early Stopping
        project='runs/ensemble_training',
        name=f'yolo26m_fold_{fold_idx}',
        device=0,
        workers=4,
        augment=True,
        
        # === Улучшенные параметры ===
        box=7.5,                        # вес лосса для боксов (очень важно!)
        cls=0.5,
        dfl=1.5,
        cos_lr=True,                    # cosine learning rate schedule
        close_mosaic=10,                 # выключаем mosaic в конце
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.3,
        save_period=10,                 # чекпоинты каждые 10 эпох
        pretrained=True,

        # multi_scale=0.25
    )
    
    # Глобальная очистка
    del yolo_model
    free_memory()

In [15]:
def custom_optimizer_step(self):
    """Кастомный шаг оптимизатора с жестким клиппингом градиентов для RT-DETR"""
    self.scaler.unscale_(self.optimizer)
    
    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
    
    self.scaler.step(self.optimizer)
    self.scaler.update()
    self.optimizer.zero_grad()
    
    if self.ema:
        self.ema.update(self.model)

def on_train_start_add_clip(trainer):
    trainer.optimizer_step = types.MethodType(custom_optimizer_step, trainer)
    print("ВНИМАНИЕ: Активирован кастомный Gradient Clipping (max_norm=0.1)!")

def train_rtdetrv1_model(fold_idx, fold_dir):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))

    print(f"--- Обучение RT-DETR на Фолде {fold_idx} ---")
    rtdetr_model = RTDETR('rtdetr-l.pt')
    
    rtdetr_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)
    rtdetr_model.add_callback("on_train_start", on_train_start_add_clip)

    rtdetr_model.train(
        data=yaml_path,
        epochs=EPOCHS_DETR,
        batch=BATCH_SIZE,
        imgsz=640,
        patience=15,                    # Early Stopping
        project='runs/ensemble_training',
        name=f'rtdetrv1_fold_{fold_idx}',
        device=0,
        workers=4,
        augment=True,
        
        amp=True,

        save_period=10,
        pretrained=True,

        box=5.0,                        
        cls=1.0,
        dfl=1.5,

        optimizer='AdamW',  
        lr0=0.0001,
        cos_lr=True,

        fliplr=0.5,         # Отзеркаливание по горизонтали 
        flipud=0.0,         # Отзеркаливание по вертикали 
        degrees=30.0,        # Вращение 
        translate=0.2,      # Сдвиг 
        scale=0.6,          # Масштаб (+-60%)
        shear=20.0,          # Сдвиг/Перекос 
        perspective=0.0001,
        
        hsv_h=0.015,        # Сдвиг оттенка. 
        hsv_s=0.3,          # Насыщенность (70%). 
        hsv_v=0.3,          # Яркость (50%). 
        bgr=0.0,            # Переворот каналов BGR->RGB 

        close_mosaic=10,
        mosaic=0.5,
        mixup=0.15,
        copy_paste=0.3,
        erasing=0.4,
       
        multi_scale=0.25
    )
    
    del rtdetr_model
    free_memory()

In [16]:
def custom_optimizer_step(self):
    """Кастомный шаг оптимизатора с жестким клиппингом градиентов для RT-DETR"""
    self.scaler.unscale_(self.optimizer)
    
    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
    
    self.scaler.step(self.optimizer)
    self.scaler.update()
    self.optimizer.zero_grad()
    
    if self.ema:
        self.ema.update(self.model)

def on_train_start_add_clip(trainer):
    trainer.optimizer_step = types.MethodType(custom_optimizer_step, trainer)
    print("ВНИМАНИЕ: Активирован кастомный Gradient Clipping (max_norm=0.1)!")

def train_rtdetrv2_model(fold_idx, fold_dir):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))

    print(f"--- Обучение RT-DETR на Фолде {fold_idx} ---")
    rtdetr_model = RTDETR('rtdetrv2-l.pth')
    
    rtdetr_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)
    rtdetr_model.add_callback("on_train_start", on_train_start_add_clip)

    rtdetr_model.train(
        data=yaml_path,
        epochs=EPOCHS_DETR,
        batch=BATCH_SIZE,
        imgsz=640,
        patience=15,                    # Early Stopping
        project='runs/ensemble_training',
        name=f'rtdetrv2_fold_{fold_idx}',
        device=0,
        workers=4,
        augment=True,

        amp=True,

        save_period=10,
        pretrained=True,

        box=5.0,                        
        cls=1.0,
        dfl=1.5,

        optimizer='AdamW',  
        lr0=0.0001,
        cos_lr=True,

        fliplr=0.5,         # Отзеркаливание по горизонтали 
        flipud=0.0,         # Отзеркаливание по вертикали 
        degrees=30.0,        # Вращение 
        translate=0.2,      # Сдвиг 
        scale=0.6,          # Масштаб (+-60%)
        shear=20.0,          # Сдвиг/Перекос 
        perspective=0.0001,
        
        hsv_h=0.015,        # Сдвиг оттенка. 
        hsv_s=0.3,          # Насыщенность (70%). 
        hsv_v=0.3,          # Яркость (50%). 
        bgr=0.0,            # Переворот каналов BGR->RGB 

        close_mosaic=10,
        mosaic=0.5,
        mixup=0.15,
        copy_paste=0.3,
        erasing=0.4,
       
        multi_scale=0.25
    )
    
    
    del rtdetr_model
    free_memory()

In [17]:
def main():
    torch.backends.cudnn.benchmark = True          
    torch.set_float32_matmul_precision('high')     
    free_memory()
    
    for fold_idx in range(1, NUM_FOLDS + 1):
        fold_dir = os.path.join(FOLDS_DIR, f'fold_{fold_idx}')
        
        print(f"\n=========================================")
        print(f" Запуск обучения для фолда {fold_idx}")
        print(f"=========================================\n")
        
        # 1. YOLO и RT-DETR
        # train_yolo11_model(fold_idx, fold_dir)

        train_yolo26_model(fold_idx, fold_dir)

        # train_rtdetrv1_model(fold_idx, fold_dir)

        # train_rtdetrv2_model(fold_idx, fold_dir)
        
        # 2. Faster R-CNN
        # train_faster_rcnn(fold_idx, fold_dir)
        
        # Мощная чистка при переходе к следующему фолду
        print(f"Фолд {fold_idx} завершен. Выполняется полная очистка кэша...")
        free_memory()
        
    print("Обучение всех моделей на всех фолдах завершено!")

In [18]:
if __name__ == '__main__':
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    main()


 Запуск обучения для фолда 1

--- Обучение YOLO11m на Фолде 1 ---
Ultralytics 8.4.21  Python-3.13.7 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=d:\LabsMIET\ML-Detection-object\dataset_folds\fold_1\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26m_

KeyboardInterrupt: 